# Arbitrage des codes représentants CRM / ERP
Entrée : `anomalie_representant_dict[<société>]` → un DataFrame par société (SICDA, PHILEA, CMGP, CAS).
Étapes : **1** config → **2** entrée → **3** sauvegarde → **4** arbitrage → **5** audit/anomalies → **6** connexions → **7** mise à jour (dry-run) → **8** rollback.

In [ ]:
import re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

# ---------------- Configuration ----------------
REGEX_CODE = r"[A-Z]{2}[0-9]{5}"            # = ^[A-Z]{2}[0-9]{5}$ (utilisé avec fullmatch)
# 2 connexions ERP : ERP_PRINCIPAL (SICDA, PHILEA, CMGP) et ERP_CAS (CAS) ; 1 connexion CRM
ERP_GROUPE_PAR_SOCIETE = {"SICDA": "ERP_PRINCIPAL", "PHILEA": "ERP_PRINCIPAL",
                          "CMGP": "ERP_PRINCIPAL", "CAS": "ERP_CAS"}
SOCIETES_ERP = set(ERP_GROUPE_PAR_SOCIETE)   # liste blanche des schémas ERP
DOSSIER_RUNS = Path("runs")

# CRM
CRM_TABLE, CRM_COL_REP, CRM_COL_CLIENT = "tiers_societe", "GST_AGENT_REPRESENTANT", "ID_CODE_TIERS_ERP_1"
CRM_COL_SOCIETE = "ID_CODE_SOCIETE"   # colonne société de tiers_societe (filtre société + client)

# ERP : pas de colonne société, la société = le schéma ({ID_CODE_SOCIETE}.bpcustomer)
ERP_TABLE, ERP_COL_REP, ERP_COL_CLIENT = "bpcustomer", "rep_0", "bpcnum_0"

COLS_ENTREE = ["ID_CODE_SOCIETE", "ID_CODE_TIERS_ERP_1", "LIB",
               "CODE_REPRESENTANT_CRM", "NOM_REPRESENTANT_CRM", "TYPE_CODE_REPRESENTANT_CRM",
               "CODE_REPRESENTANT_ERP", "NOM_REPRESENTANT_ERP"]
SRC_CRM, SRC_ERP, SRC_ANOMALIE, SRC_NON_TRAITE = "CRM", "ERP", "ANOMALIE_FORMAT", "NON_TRAITE"

## 2. Données d'entrée : `anomalie_representant_dict`

In [ ]:
# anomalie_representant_dict doit déjà exister dans votre session (clé = société, valeur = DataFrame)

def preparer_entree(d: dict) -> pd.DataFrame:
    """Valide les clés/colonnes, force ID_CODE_SOCIETE = clé du dict, concatène tout."""
    frames = []
    for cle, df in d.items():
        soc = str(cle).strip().upper()
        if soc not in SOCIETES_ERP:
            raise ValueError(f"Société inconnue dans le dict : {cle!r} (attendu : {sorted(SOCIETES_ERP)})")
        manq = set(COLS_ENTREE) - set(df.columns) - {"ID_CODE_SOCIETE"}
        if manq:
            raise ValueError(f"[{soc}] colonnes manquantes : {sorted(manq)}")
        f = df.copy()
        f["ID_CODE_SOCIETE"] = soc          # la clé du dict fait foi (utilisée pour le schéma ERP)
        frames.append(f[COLS_ENTREE])
    return pd.concat(frames, ignore_index=True)

df_entree = preparer_entree(anomalie_representant_dict)
print(df_entree.groupby("ID_CODE_SOCIETE").size())
df_entree

## 3. Sauvegarde des données d'origine

In [ ]:
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
DOSSIER = DOSSIER_RUNS / RUN_ID
DOSSIER.mkdir(parents=True, exist_ok=True)

df_entree.to_csv(DOSSIER / "backup_entree.csv", index=False, encoding="utf-8-sig")
print("Sauvegarde :", DOSSIER / "backup_entree.csv", "| RUN_ID =", RUN_ID)

## 4. Arbitrage (calcul uniquement, aucune écriture en base)

In [ ]:
def _est_vide(s):  return s.isna() | s.astype("string").str.strip().fillna("").eq("")
def _est_valide(s): return s.astype("string").str.fullmatch(REGEX_CODE).fillna(False).astype(bool)

def arbitrer(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    crm_code, erp_code = out["CODE_REPRESENTANT_CRM"], out["CODE_REPRESENTANT_ERP"]
    crm_vide, erp_vide = _est_vide(out["NOM_REPRESENTANT_CRM"]), _est_vide(out["NOM_REPRESENTANT_ERP"])
    crm_ok, erp_ok = _est_valide(crm_code), _est_valide(erp_code)

    deux_vides = crm_vide & erp_vide
    crm_seul, erp_seul = ~crm_vide & erp_vide, crm_vide & ~erp_vide
    deux = ~crm_vide & ~erp_vide

    regles = [
        ("R1_NOMS_VIDES",           deux_vides,                  SRC_NON_TRAITE),
        ("R2_CRM_SEUL_OK",          crm_seul & crm_ok,           SRC_CRM),
        ("R2_CRM_SEUL_KO",          crm_seul & ~crm_ok,          SRC_ANOMALIE),
        ("R2_ERP_SEUL_OK",          erp_seul & erp_ok,           SRC_ERP),
        ("R2_ERP_SEUL_KO",          erp_seul & ~erp_ok,          SRC_ANOMALIE),
        ("R3_DEUX_OK_PRIORITE_ERP", deux & crm_ok & erp_ok,      SRC_ERP),
        ("R3_SEUL_CRM_OK",          deux & crm_ok & ~erp_ok,     SRC_CRM),
        ("R3_SEUL_ERP_OK",          deux & ~crm_ok & erp_ok,     SRC_ERP),
        ("R3_AUCUN_OK",             deux & ~crm_ok & ~erp_ok,    SRC_ANOMALIE),
    ]
    conds = [r[1].to_numpy() for r in regles]
    out["REGLE_APPLIQUEE"] = np.select(conds, [r[0] for r in regles], default="INCONNUE")
    out["SOURCE_VERITE_ARBITREE"] = np.select(conds, [r[2] for r in regles], default=SRC_NON_TRAITE)

    src = out["SOURCE_VERITE_ARBITREE"]
    out["CODE_REPRESENTANT_ARBITRE"] = np.select(
        [src.eq(SRC_CRM).to_numpy(), src.eq(SRC_ERP).to_numpy()],
        [crm_code.to_numpy(dtype=object), erp_code.to_numpy(dtype=object)], default=None).astype(object)
    out.loc[src.isin([SRC_ANOMALIE, SRC_NON_TRAITE]), "CODE_REPRESENTANT_ARBITRE"] = None

    resolu = out["CODE_REPRESENTANT_ARBITRE"].notna()
    out["MAJ_CRM_NECESSAIRE"] = resolu & out["CODE_REPRESENTANT_ARBITRE"].ne(crm_code)
    out["MAJ_ERP_NECESSAIRE"] = resolu & out["CODE_REPRESENTANT_ARBITRE"].ne(erp_code)
    out["DATE_ARBITRAGE"] = datetime.now().isoformat(timespec="seconds")
    return out

df_arbitre = arbitrer(df_entree)

# Résultats redécoupés par société, même structure que l'entrée
resultat_dict = {soc: g.reset_index(drop=True) for soc, g in df_arbitre.groupby("ID_CODE_SOCIETE")}

print(df_arbitre["SOURCE_VERITE_ARBITREE"].value_counts())
df_arbitre[["ID_CODE_SOCIETE", "ID_CODE_TIERS_ERP_1", "CODE_REPRESENTANT_ARBITRE",
            "SOURCE_VERITE_ARBITREE", "REGLE_APPLIQUEE"]]

## 5. Audit log + anomalies

In [ ]:
audit = pd.DataFrame({
    "RUN_ID": RUN_ID,
    "ID_CODE_SOCIETE": df_arbitre["ID_CODE_SOCIETE"],
    "ID_CODE_TIERS_ERP_1": df_arbitre["ID_CODE_TIERS_ERP_1"],
    "ANCIEN_CODE_CRM": df_arbitre["CODE_REPRESENTANT_CRM"],
    "ANCIEN_CODE_ERP": df_arbitre["CODE_REPRESENTANT_ERP"],
    "CODE_RETENU": df_arbitre["CODE_REPRESENTANT_ARBITRE"],
    "SOURCE_RETENUE": df_arbitre["SOURCE_VERITE_ARBITREE"],
    "REGLE_APPLIQUEE": df_arbitre["REGLE_APPLIQUEE"],
    "HORODATAGE": df_arbitre["DATE_ARBITRAGE"],
})
anomalies = df_arbitre[df_arbitre["SOURCE_VERITE_ARBITREE"].isin([SRC_ANOMALIE, SRC_NON_TRAITE])].copy()

audit.to_csv(DOSSIER / "audit_arbitrage.csv", index=False, encoding="utf-8-sig")
anomalies.to_csv(DOSSIER / "anomalies.csv", index=False, encoding="utf-8-sig")
print(f"Audit : {len(audit)} lignes | Anomalies : {len(anomalies)} lignes")
anomalies

## 6. Connexions (à renseigner) : 3 au total
- **CRM** : `crm_engine`
- **ERP principal** : `erp_engine` → schémas SICDA, PHILEA, CMGP
- **ERP CAS** : `erp_cas_engine` → schéma CAS

In [ ]:
# crm_engine     = create_engine("<URL_CRM>")
# erp_engine     = create_engine("<URL_ERP_SICDA_PHILEA_CMGP>")
# erp_cas_engine = create_engine("<URL_ERP_CAS>")
crm_engine = erp_engine = erp_cas_engine = None   # à remplacer

erp_engines = {"ERP_PRINCIPAL": erp_engine, "ERP_CAS": erp_cas_engine}

## 7. Mise à jour des bases (étape ultérieure, après validation)
Une transaction par connexion : CRM, ERP principal (SICDA/PHILEA/CMGP), ERP CAS.
`dry_run=True` par défaut : simule et écrit `update_log.csv` sans rien modifier.

In [ ]:
def _sql_crm():
    filtre = f" AND {CRM_COL_SOCIETE} = :societe" if CRM_COL_SOCIETE else ""
    return f"UPDATE {CRM_TABLE} SET {CRM_COL_REP} = :val WHERE {CRM_COL_CLIENT} = :client{filtre}"

def _sql_erp(societe):
    if societe not in SOCIETES_ERP:
        raise ValueError(f"Société ERP non autorisée : {societe!r}")
    return f"UPDATE {societe}.{ERP_TABLE} SET {ERP_COL_REP} = :val WHERE {ERP_COL_CLIENT} = :client"

def appliquer_mises_a_jour(df, run_id, crm_engine, erp_engines, dry_run=True):
    log, horo = [], datetime.now().isoformat(timespec="seconds")
    def _l(systeme, connexion, r, ancien, statut, rc):
        return {"RUN_ID": run_id, "SYSTEME": systeme, "CONNEXION": connexion,
                "ID_CODE_SOCIETE": r.ID_CODE_SOCIETE, "ID_CODE_TIERS_ERP_1": r.ID_CODE_TIERS_ERP_1,
                "ANCIENNE_VALEUR": ancien, "NOUVELLE_VALEUR": r.CODE_REPRESENTANT_ARBITRE,
                "STATUT": statut, "ROWCOUNT": rc, "HORODATAGE": horo}
    try:
        # CRM : une transaction
        l_crm = df[df["MAJ_CRM_NECESSAIRE"]]
        if len(l_crm):
            local = []
            if dry_run:
                local = [_l("CRM", "CRM", r, r.CODE_REPRESENTANT_CRM, "DRY_RUN", None) for r in l_crm.itertuples()]
            else:
                with crm_engine.begin() as conn:
                    for r in l_crm.itertuples():
                        p = {"val": r.CODE_REPRESENTANT_ARBITRE, "client": r.ID_CODE_TIERS_ERP_1}
                        if CRM_COL_SOCIETE: p["societe"] = r.ID_CODE_SOCIETE
                        rc = conn.execute(text(_sql_crm()), p).rowcount
                        local.append(_l("CRM", "CRM", r, r.CODE_REPRESENTANT_CRM, "APPLIQUE", rc))
            log.extend(local)   # ajouté au log seulement si la transaction a été validée

        # ERP : une transaction PAR connexion (ERP_PRINCIPAL, ERP_CAS)
        l_erp = df[df["MAJ_ERP_NECESSAIRE"]]
        for groupe, sous in l_erp.groupby(l_erp["ID_CODE_SOCIETE"].map(ERP_GROUPE_PAR_SOCIETE)):
            local = []
            if dry_run:
                local = [_l("ERP", groupe, r, r.CODE_REPRESENTANT_ERP, "DRY_RUN", None) for r in sous.itertuples()]
            else:
                with erp_engines[groupe].begin() as conn:
                    for r in sous.itertuples():
                        p = {"val": r.CODE_REPRESENTANT_ARBITRE, "client": r.ID_CODE_TIERS_ERP_1}
                        rc = conn.execute(text(_sql_erp(r.ID_CODE_SOCIETE)), p).rowcount
                        local.append(_l("ERP", groupe, r, r.CODE_REPRESENTANT_ERP, "APPLIQUE", rc))
            log.extend(local)
    finally:
        # le log est écrit même en cas d'erreur : il liste ce qui a réellement été validé
        pd.DataFrame(log).to_csv(DOSSIER_RUNS / run_id / "update_log.csv", index=False, encoding="utf-8-sig")
    return pd.DataFrame(log)

# 1) Simulation
log_maj = appliquer_mises_a_jour(df_arbitre, RUN_ID, crm_engine, erp_engines, dry_run=True)
log_maj

In [ ]:
# 2) Application réelle (décommenter après validation du dry-run)
# log_maj = appliquer_mises_a_jour(df_arbitre, RUN_ID, crm_engine, erp_engines, dry_run=False)
# log_maj

## 8. Rollback (cas de secours)
Restaure dans CRM et ERP les valeurs **d'origine** (lues dans `backup_entree.csv`) des lignes réellement modifiées par le run.
Garde de sécurité : une ligne n'est restaurée que si sa valeur actuelle est toujours le code arbitré (sinon `IGNORE_VALEUR_MODIFIEE`). `forcer=True` désactive la garde.

In [ ]:
def rollback(run_id, crm_engine, erp_engines, dry_run=True, forcer=False):
    d = DOSSIER_RUNS / run_id
    log = pd.read_csv(d / "update_log.csv", dtype=str, keep_default_na=False, na_values=[""])
    bk = (pd.read_csv(d / "backup_entree.csv", dtype=str, keep_default_na=False, na_values=[""])
            .drop_duplicates(["ID_CODE_SOCIETE", "ID_CODE_TIERS_ERP_1"])
            .set_index(["ID_CODE_SOCIETE", "ID_CODE_TIERS_ERP_1"]))
    a_annuler = log[log["STATUT"] == "APPLIQUE"]
    res = []

    def garde(sql, col):
        return sql + f" AND (({col} = :attendu) OR ({col} IS NULL AND :attendu IS NULL))"

    # une transaction par connexion : CRM, ERP_PRINCIPAL, ERP_CAS
    for connexion, sous in a_annuler.groupby("CONNEXION"):
        systeme = sous["SYSTEME"].iloc[0]
        col_orig = "CODE_REPRESENTANT_CRM" if systeme == "CRM" else "CODE_REPRESENTANT_ERP"
        col_rep = CRM_COL_REP if systeme == "CRM" else ERP_COL_REP
        taches = []
        for r in sous.itertuples():
            orig = bk.loc[(r.ID_CODE_SOCIETE, r.ID_CODE_TIERS_ERP_1), col_orig]
            orig = None if pd.isna(orig) else orig
            sql = _sql_crm() if systeme == "CRM" else _sql_erp(r.ID_CODE_SOCIETE)
            if not forcer: sql = garde(sql, col_rep)
            p = {"val": orig, "client": r.ID_CODE_TIERS_ERP_1, "attendu": r.NOUVELLE_VALEUR}
            if systeme == "CRM" and CRM_COL_SOCIETE: p["societe"] = r.ID_CODE_SOCIETE
            taches.append((r, sql, p, orig))

        def _enreg(r, orig, statut, rc):
            res.append({"RUN_ID": run_id, "SYSTEME": systeme, "CONNEXION": connexion,
                        "ID_CODE_SOCIETE": r.ID_CODE_SOCIETE, "ID_CODE_TIERS_ERP_1": r.ID_CODE_TIERS_ERP_1,
                        "VALEUR_RESTAUREE": orig, "STATUT": statut, "ROWCOUNT": rc,
                        "HORODATAGE": datetime.now().isoformat(timespec="seconds")})
        if dry_run:
            for r, _, _, orig in taches: _enreg(r, orig, "DRY_RUN", None)
        else:
            engine = crm_engine if systeme == "CRM" else erp_engines[connexion]
            with engine.begin() as conn:
                for r, sql, p, orig in taches:
                    rc = conn.execute(text(sql), p).rowcount
                    _enreg(r, orig, "RESTAURE" if rc else "IGNORE_VALEUR_MODIFIEE", rc)

    df_res = pd.DataFrame(res)
    df_res.to_csv(d / f"rollback_log_{datetime.now():%Y%m%d_%H%M%S}.csv", index=False, encoding="utf-8-sig")
    return df_res

# Simulation du rollback
rollback(RUN_ID, crm_engine, erp_engines, dry_run=True)

In [ ]:
# Rollback réel (à décommenter uniquement en cas de secours)
# rollback(RUN_ID, crm_engine, erp_engines, dry_run=False)

In [ ]:
# Reset des colonnes d'arbitrage côté DataFrame (n'agit pas sur les bases)
COLS_ARBITRAGE = ["CODE_REPRESENTANT_ARBITRE", "SOURCE_VERITE_ARBITREE", "REGLE_APPLIQUEE",
                  "DATE_ARBITRAGE", "MAJ_CRM_NECESSAIRE", "MAJ_ERP_NECESSAIRE"]
# df_arbitre = df_arbitre.drop(columns=COLS_ARBITRAGE)